# eltq demo

ELT secondary-uncertainty quantization: raw vs beta-secondary loss-grid
distributions, the SciPy reference comparison, a 1M-row run on a 2^16-node
grid, determinism, and memory-mapped output.

Launch from the repository root:

```powershell
uv run --with jupyter --with matplotlib jupyter lab
```

`uv run` picks up the project environment (the eltq wheel plus the dev
group with scipy/pandas); `--with` adds jupyter and matplotlib ephemerally
without touching the project dependencies.

In [ ]:
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import eltq
from eltq import reference  # pure SciPy twin; also the automatic fallback

ROOT = Path.cwd() if (Path.cwd() / "fixtures").exists() else Path.cwd().parent
print(f"eltq {eltq.__version__}, backend: {eltq.BACKEND}")

## One event

Without secondary uncertainty an event is an atom of rate $\lambda$ at its
mean loss $\mu$. With it, the conditional loss is a beta on $[0, M]$
matching $(\mu, \sigma^2)$, approximated by $K$ equally weighted midpoint
quantiles and deposited onto the grid with the mass- and mean-preserving
linear rule.

In [ ]:
raw, secondary, meta = eltq.quantize_arrays(
    [0.1], [100.0], [40.0], [25.0], [0.0],
    bucket_size=1.0, quadrature_points=32)
grid = meta.bucket_size * np.arange(meta.grid_length)

fig, ax = plt.subplots(figsize=(8.5, 3.2))
ax.bar(grid, secondary, width=1.0, color="steelblue",
       label="secondary: 32 beta midpoint quantiles")
ax.axvline(40.0, color="firebrick", ls="--",
           label="raw: single atom of rate 0.1 at $\\mu$ = 40")
ax.set(xlabel="loss", ylabel="rate weight",
       title="One event: M = 100, $\\mu$ = 40, $\\sigma$ = 25, K = 32")
ax.legend()
plt.tight_layout()
print(f"masses: raw {meta.raw_mass:.6f}, secondary {meta.secondary_mass:.6f}"
      f"  (total rate {meta.total_rate})")
print(f"means:  raw {meta.raw_mean:.4f}, secondary {meta.secondary_mean:.4f}"
      f"  (finite-K mean error {meta.secondary_mean_error:.2e})")

## A portfolio through the CSV route

The committed 5000-row synthetic fixture, parsed and quantized in Rust,
with per-category row diagnostics.

In [ ]:
raw, secondary, meta = eltq.quantize_csv(
    ROOT / "fixtures" / "medium_elt.csv", bucket_size=5000.0)
grid = meta.bucket_size * np.arange(meta.grid_length)

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.semilogy(grid, raw, lw=0.8, color="firebrick", label="raw")
ax.semilogy(grid, secondary, lw=0.8, color="steelblue", label="secondary")
ax.set(xlabel="loss", ylabel="rate weight (log)",
       title=f"5000-row fixture: K = 32, h = 5000, n = {meta.grid_length}")
ax.legend()
plt.tight_layout()
print(f"total rate {meta.total_rate:.6f}, AAL {meta.analytic_aal:,.0f}")
for category, issues in meta.diagnostics.items():
    print(f"  {category:>16}: {issues.count:>4}  e.g. {issues.examples[:3]}")

## Comparison with the raw SciPy reference

`eltq.reference.quantize_reference` is the transparent pure-Python/SciPy
implementation of the same contract. It doubles as the automatic fallback:
if the compiled extension is not importable, `import eltq` serves the whole
API through it (`eltq.BACKEND == "reference"`), so code written against
eltq still runs — just slower.

In [ ]:
import pandas as pd

df = pd.read_csv(ROOT / "fixtures" / "medium_elt.csv",
                 float_precision="round_trip")
cols = tuple(df[c].to_numpy() for c in
             ("rate", "max_loss", "mean_loss", "sd_indep", "sd_corr"))

t0 = time.perf_counter()
raw_rust, sec_rust, meta_rust = eltq.quantize_arrays(*cols, bucket_size=5000.0)
t_rust = time.perf_counter() - t0

t0 = time.perf_counter()
raw_ref, sec_ref, meta_ref = reference.quantize_reference(*cols, bucket_size=5000.0)
t_ref = time.perf_counter() - t0

scale = meta_ref.total_rate
print(f"raw agree:       {np.allclose(raw_rust, raw_ref, rtol=1e-9, atol=1e-13 * scale)}")
print(f"secondary agree: {np.allclose(sec_rust, sec_ref, rtol=1e-9, atol=1e-13 * scale)}")
print(f"eltq {t_rust * 1e3:7.1f} ms   scipy reference {t_ref:6.2f} s   "
      f"speedup {t_ref / t_rust:,.0f}x   (5000 rows, K = 32)")

## One million rows on a 2^16-node grid

Grid length auto-derives as $\lceil \max M / h \rceil + 1$; to target
exactly $2^{16}$ nodes, pass `grid_length=2**16` with any $h$ satisfying
$(2^{16}-1)\,h \ge \max M$ — here $h = 80$ covers $65535 \times 80 =
5.24\text{e}6 \ge 5\text{e}6$. Grid length is irrelevant to speed: each
deposit touches exactly two nodes whatever $n$ is.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "benchmarks"))
from run_end_to_end import synthetic_elt, cols_tuple

elt_1m = synthetic_elt(1_000_000)
t0 = time.perf_counter()
raw, secondary, meta = eltq.quantize_arrays(
    *cols_tuple(elt_1m), bucket_size=80.0, grid_length=2**16,
    quadrature_points=32)
elapsed = time.perf_counter() - t0

print(f"{elapsed:.2f} s -> {1_000_000 / elapsed:,.0f} rows/s "
      f"on a {meta.grid_length:,}-node grid (K = 32)")
print(f"mass error |raw_mass - total_rate|: "
      f"{abs(meta.raw_mass - meta.total_rate):.3e}")
print(f"raw mean == analytic AAL: "
      f"{np.isclose(meta.raw_mean, meta.analytic_aal, rtol=1e-10)}")

# SciPy reference on a 20k subsample, extrapolated to the full input.
sub = {k: v[:20_000] for k, v in elt_1m.items()}
t0 = time.perf_counter()
reference.quantize_reference(*cols_tuple(sub), bucket_size=80.0,
                             grid_length=2**16, quadrature_points=32)
t_ref = time.perf_counter() - t0
full_ref = t_ref * 50
print(f"scipy reference: {t_ref:.1f} s for 20k rows -> ~{full_ref / 60:.0f} min "
      f"for 1M; eltq speedup ~{full_ref / elapsed:,.0f}x")

In [ ]:
grid = meta.bucket_size * np.arange(meta.grid_length)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.semilogy(grid, raw, lw=0.5, color="firebrick", alpha=0.8, label="raw")
ax.semilogy(grid, secondary, lw=0.5, color="steelblue", alpha=0.8,
            label="secondary")
ax.set(xlabel="loss", ylabel="rate weight (log)",
       title="1M rows, K = 32, n = $2^{16}$")
ax.legend()
plt.tight_layout()

## Determinism

Fixed input and config give bitwise-identical outputs for any thread count
(private per-chunk histograms folded sequentially in chunk order).

In [ ]:
sub = {k: v[:200_000] for k, v in elt_1m.items()}
r1, s1, _ = eltq.quantize_arrays(*cols_tuple(sub), bucket_size=80.0,
                                 grid_length=2**16, threads=1)
r2, s2, _ = eltq.quantize_arrays(*cols_tuple(sub), bucket_size=80.0,
                                 grid_length=2**16, threads=None)
print("bitwise identical, 1 thread vs all cores:",
      np.array_equal(r1, r2) and np.array_equal(s1, s2))

## Memory-mapped output

`quantize_to_npy` writes a transactional `(2, n)` `.npy` (row 0 raw, row 1
secondary) plus adjacent JSON metadata, reopenable without loading.

In [ ]:
out = ROOT / "benchmarks" / "scratch" / "demo_grids.npy"
out.parent.mkdir(parents=True, exist_ok=True)
meta_npy = eltq.quantize_to_npy(cols_tuple(elt_1m), out,
                                bucket_size=80.0, grid_length=2**16)
grids = np.load(out, mmap_mode="r")
print(grids.shape, grids.dtype, "-", out)
print("raw mass from the memmap:", float(grids[0].sum()))